# Creacion ruta entre Punto A y B

In [1]:
import arcpy
arcpy.env.overwriteOutput = True

# ─── PARAMETROS A COMPLETAR ───────────────────────────────────────────────────
    # ── Coordenadas ───────────────────────────────────────────────────────────
origen_coord  = (-3.8367148, 40.2991133)   # Loranca, Fuenlabrada
destino_coord = (-3.5708900, 40.4638900)   # T1 Aeropuerto Madrid-Barajas

    # ── Etiqueta del vehículo ─────────────────────────────────────────────────
ETIQUETA = "SIN"   # Opciones: "0", "ECO", "C", "B", "SIN"
# ──────────────────────────────────────────────────────────────────────────────

# ─── CONFIGURACIÓN ───────────────────────────────────────────────────────────
network     = r"C:\EsriTraining\TFM\GDB\GDB_Principal.gdb\Red_Viaria\ND_Madrid_Notebook"
gdb         = r"C:\EsriTraining\TFM\GDB\GDB_Principal.gdb"
fd_rutas    = gdb + r"\Rutas_Punto_A_Punto_B"   # ← Feature Dataset de salida
stops_table = fd_rutas + r"\Stops_Ruta"

# ── Capas ZBE ─────────────────────────────────────────────────────────────────
zbe_general  = gdb + r"\ZBE\ZBE_Con_Acceso_CON_Pegatina"
zbe_centro   = gdb + r"\ZBE\ZBEDEP_Distrito_Centro_Limite"
zbe_eliptica = gdb + r"\ZBE\ZBEDEP_Plaza_Eliptica_Limite"

# ─── LÓGICA DE RESTRICCIONES ─────────────────────────────────────────────────
restricciones = {
    "0":   {"general": False, "centro": False, "eliptica": False},
    "ECO": {"general": False, "centro": False, "eliptica": False},
    "C":   {"general": False, "centro": True,  "eliptica": True },
    "B":   {"general": False, "centro": True,  "eliptica": True },
    "SIN": {"general": True,  "centro": True,  "eliptica": True },
}

reglas = restricciones[ETIQUETA]
print(f"\n🚗 Etiqueta: {ETIQUETA}")
print(f"   ZBE General   → {'❌ Prohibido' if reglas['general']  else '✅ Libre'}")
print(f"   ZBEDEP Centro → {'❌ Prohibido' if reglas['centro']   else '✅ Libre'}")
print(f"   Plaza Elípt.  → {'❌ Prohibido' if reglas['eliptica'] else '✅ Libre'}")

sr_red   = arcpy.Describe(network).spatialReference
sr_wgs84 = arcpy.SpatialReference(4326)

# ─── LIMPIAR CAPAS RESIDUALES DEL MAPA ───────────────────────────────────────
try:
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    mapa = aprx.activeMap
    for lyr in mapa.listLayers():
        if lyr.name in ["Ruta1_SinZBE", "Ruta2_EvitaZBE"]:
            mapa.removeLayer(lyr)
except:
    pass

# ─── 1. CONSTRUIR RED ────────────────────────────────────────────────────────
print("\nConstruyendo red...")
arcpy.na.BuildNetwork(network)
print("✅ Red construida")

# ─── 2. STOPS ────────────────────────────────────────────────────────────────
if arcpy.Exists(stops_table):
    arcpy.management.Delete(stops_table)
arcpy.management.CreateFeatureclass(fd_rutas, "Stops_Ruta", "POINT", spatial_reference=sr_red)
with arcpy.da.InsertCursor(stops_table, ["SHAPE@"]) as cur:
    for x, y in [origen_coord, destino_coord]:
        cur.insertRow([arcpy.PointGeometry(arcpy.Point(x, y), sr_wgs84)])
print(f"✅ Stops creados: {int(arcpy.management.GetCount(stops_table)[0])}")

# ─── 3. TRAVEL MODE ──────────────────────────────────────────────────────────
tm = arcpy.na.GetTravelModes(network)["Conduccion"]

# ─── 4. RUTA 1 — Sin restricciones ZBE ───────────────────────────────────────
print("\nCalculando Ruta 1 (sin restricciones ZBE)...")
rl1 = arcpy.na.MakeRouteAnalysisLayer(
    network_data_source=network,
    layer_name="Ruta1_SinZBE",
    travel_mode=tm,
    line_shape="ALONG_NETWORK"
).getOutput(0)
arcpy.na.AddLocations(rl1, "Stops", stops_table, search_tolerance="1000 Meters")
arcpy.na.Solve(rl1, ignore_invalids="SKIP", terminate_on_solve_error="CONTINUE")

n_r1 = int(arcpy.management.GetCount(rl1.listLayers("Routes")[0])[0])
if n_r1 == 0:
    print("❌ Sin ruta base")
    raise SystemExit()

ruta1_fc = fd_rutas + r"\Ruta1_SinZBE"
arcpy.management.CopyFeatures(rl1.listLayers("Routes")[0], ruta1_fc)
print("✅ Ruta 1 guardada")

# ─── 5. COMPROBAR INTERSECCIONES ─────────────────────────────────────────────
def cruza_zona(ruta_fc, zona_fc, nombre):
    tmp = gdb + r"\Tmp_Intersect"
    if arcpy.Exists(tmp): arcpy.management.Delete(tmp)
    arcpy.analysis.Intersect([ruta_fc, zona_fc], tmp)
    resultado = int(arcpy.management.GetCount(tmp)[0]) > 0
    print(f"   {nombre}: {'⚠️  SÍ cruza' if resultado else '✅ No cruza'}")
    arcpy.management.Delete(tmp)
    return resultado

print("\nComprobando intersecciones...")
cruza_gral     = cruza_zona(ruta1_fc, zbe_general,  "ZBE General  ")
cruza_centro   = cruza_zona(ruta1_fc, zbe_centro,   "ZBEDEP Centro")
cruza_eliptica = cruza_zona(ruta1_fc, zbe_eliptica, "Plaza Elípt. ")

# ─── 6. ¿ES VÁLIDA LA RUTA? ──────────────────────────────────────────────────
es_invalida = (
    (reglas["general"]  and cruza_gral)     or
    (reglas["centro"]   and cruza_centro)   or
    (reglas["eliptica"] and cruza_eliptica)
)

# ─── 7. CAMPO COLOR EN RUTA 1 ────────────────────────────────────────────────
arcpy.management.AddField(ruta1_fc, "Color_ZBE", "TEXT", field_length=10)
arcpy.management.AddField(ruta1_fc, "Etiqueta",  "TEXT", field_length=10)
arcpy.management.CalculateField(ruta1_fc, "Color_ZBE", "'ROJO'" if es_invalida else "'VERDE'")
arcpy.management.CalculateField(ruta1_fc, "Etiqueta",  f"'{ETIQUETA}'")

if es_invalida:
    print(f"\n⚠️  Ruta NO válida para etiqueta {ETIQUETA} → calculando alternativa...")
else:
    print(f"\n✅ Ruta válida para etiqueta {ETIQUETA} → no necesita alternativa")

# ─── 8. RUTA 2 — Evitando zonas prohibidas ───────────────────────────────────
if es_invalida:
    rl2 = arcpy.na.MakeRouteAnalysisLayer(
        network_data_source=network,
        layer_name="Ruta2_EvitaZBE",
        travel_mode=tm,
        line_shape="ALONG_NETWORK"
    ).getOutput(0)

    barreras = []
    if reglas["general"]:
        arcpy.na.AddLocations(rl2, "Polygon Barriers", zbe_general,
                              field_mappings="BarrierType 0 #")
        barreras.append("ZBE General")
    if reglas["centro"]:
        arcpy.na.AddLocations(rl2, "Polygon Barriers", zbe_centro,
                              field_mappings="BarrierType 0 #")
        barreras.append("ZBEDEP Centro")
    if reglas["eliptica"]:
        arcpy.na.AddLocations(rl2, "Polygon Barriers", zbe_eliptica,
                              field_mappings="BarrierType 0 #")
        barreras.append("Plaza Elíptica")

    print(f"   Barreras activas: {', '.join(barreras)}")

    arcpy.na.AddLocations(rl2, "Stops", stops_table, search_tolerance="1000 Meters")
    arcpy.na.Solve(rl2, ignore_invalids="SKIP", terminate_on_solve_error="CONTINUE")

    n_r2 = int(arcpy.management.GetCount(rl2.listLayers("Routes")[0])[0])
    if n_r2 > 0:
        ruta2_fc = fd_rutas + r"\Ruta2_EvitaZBE"
        arcpy.management.CopyFeatures(rl2.listLayers("Routes")[0], ruta2_fc)
        arcpy.management.AddField(ruta2_fc, "Color_ZBE", "TEXT", field_length=10)
        arcpy.management.AddField(ruta2_fc, "Etiqueta",  "TEXT", field_length=10)
        arcpy.management.CalculateField(ruta2_fc, "Color_ZBE", "'VERDE'")
        arcpy.management.CalculateField(ruta2_fc, "Etiqueta",  f"'{ETIQUETA}'")
        print("✅ Ruta alternativa guardada:", ruta2_fc)
    else:
        print("⚠️  No existe ruta que evite completamente las zonas restringidas")
        for i in range(arcpy.GetMessageCount()):
            print(f"  [{i}] {arcpy.GetMessage(i)}")

# ─── 9. RESUMEN FINAL ────────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"RESUMEN — Etiqueta {ETIQUETA} | Punto A → Punto B")
print("="*55)
print(f"Stops    : {stops_table}")
print(f"Ruta 1   ({'❌ INVÁLIDA - ROJA' if es_invalida else '✅ VÁLIDA - VERDE'}): {ruta1_fc}")
if es_invalida:
    print(f"Ruta 2   (✅ ALTERNATIVA - VERDE): {fd_rutas + chr(92) + 'Ruta2_EvitaZBE'}")


🚗 Etiqueta: SIN
   ZBE General   → ❌ Prohibido
   ZBEDEP Centro → ❌ Prohibido
   Plaza Elípt.  → ❌ Prohibido

Construyendo red...
✅ Red construida
✅ Stops creados: 2

Calculando Ruta 1 (sin restricciones ZBE)...
✅ Ruta 1 guardada

Comprobando intersecciones...
   ZBE General  : ⚠️  SÍ cruza
   ZBEDEP Centro: ✅ No cruza
   Plaza Elípt. : ✅ No cruza

⚠️  Ruta NO válida para etiqueta SIN → calculando alternativa...
   Barreras activas: ZBE General, ZBEDEP Centro, Plaza Elíptica
✅ Ruta alternativa guardada: C:\EsriTraining\TFM\GDB\GDB_Principal.gdb\Rutas_Punto_A_Punto_B\Ruta2_EvitaZBE

RESUMEN — Etiqueta SIN | Punto A → Punto B
Stops    : C:\EsriTraining\TFM\GDB\GDB_Principal.gdb\Rutas_Punto_A_Punto_B\Stops_Ruta
Ruta 1   (❌ INVÁLIDA - ROJA): C:\EsriTraining\TFM\GDB\GDB_Principal.gdb\Rutas_Punto_A_Punto_B\Ruta1_SinZBE
Ruta 2   (✅ ALTERNATIVA - VERDE): C:\EsriTraining\TFM\GDB\GDB_Principal.gdb\Rutas_Punto_A_Punto_B\Ruta2_EvitaZBE
